# Solving the ladder, one rank at a time

`01-build-the-model.ipynb` builds the model. This one solves it, and shows the mechanism the
whole recommendation rests on: **lexicographic optimisation**, where fifteen objectives are
minimised in strict priority order rather than weighted into one.

The thing to take away is that there are no weights anywhere. A lower rank cannot buy a
degradation of a higher one at any exchange rate, because by the time it is optimised the
higher rank's value is already a constraint.

In [13]:
import inspect
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Code, Markdown, display

print(f"kernel: Python {sys.version.split()[0]}")

HERE = Path.cwd()
MODEL = HERE.parent.parent / "model"
assert (MODEL / "capacity_allocator").is_dir(), (
    f"run this from output/notebooks/, not {HERE}"
)
sys.path.insert(0, str(MODEL))

from capacity_allocator import (
    config as cfg, data, model, scenarios,
)


def show_source(fn):
    src = inspect.getsource(fn)
    where = f"{Path(inspect.getsourcefile(fn)).name}:{inspect.getsourcelines(fn)[1]}"
    display(Markdown(f"`{fn.__qualname__}` &nbsp;&mdash;&nbsp; `{where}`, "
                     f"{len(src.splitlines())} lines"))
    display(Code(src, language="python"))

kernel: Python 3.12.10


---
## 1. A model to solve

`ob_backlog_build` rather than `baseline`, deliberately: it breaches a rank, so there is
something for the ladder to work on. It is also one of the three scenarios that breach by an
**identical** 22,662 units and get three different answers - the other two differ only in
which ceiling stops them.

In [14]:
scn = scenarios.get("ob_backlog_build")
p = data.build(data.generate(scn))
print(f"{scn.name}: {scn.headline}")

backend = model.make_backend(label="ladder_walkthrough")
m = model.build(backend, p, freeze_moves=False)
nv, nc = backend.report_size()
print(f"\nbuilt: {nv} variables, {nc} constraints, {len(m.tiers)} ranks")

ob_backlog_build: Outbound arrivals surge

built: 922 variables, 754 constraints, 15 ranks


---
## 2. The fifteen ranks

$$
\begin{array}{rllc}
1 & \text{Inbound storage, this shift} & \sum_{h \in \mathcal{H}_{0}} o_{h} & \text{slack}\\
2 & \text{Inbound storage, later} & \sum_{h \in \mathcal{H}_{1}} o_{h} & \text{slack}\\
3 & \text{Cut-off risk, this shift} & \sum_{c \in \mathcal{C}_{0}} g_{c} & \text{slack}\\
4 & \text{Cut-off risk, later} & \sum_{c \in \mathcal{C}_{1}} g_{c} & \text{slack}\\
5 & \text{Backlog too high, this shift} & \sum_{h \in \mathcal{H}_{0}} v^{\uparrow}_{h} & \text{slack}\\
6 & \text{Backlog too high, later} & \sum_{h \in \mathcal{H}_{1}} v^{\uparrow}_{h} & \text{slack}\\
7 & \text{End of day above target} & e^{\max}_{\mathrm{eod}} & \text{slack}\\
8 & \text{Backlog too low, this shift} & \sum_{h \in \mathcal{H}_{0}} v^{\downarrow}_{h} & \text{slack}\\
9 & \text{Backlog too low, later} & \sum_{h \in \mathcal{H}_{1}} v^{\downarrow}_{h} & \text{slack}\\
10 & \text{End of day below target} & e^{\min}_{\mathrm{eod}} & \text{slack}\\
11 & \text{Returns ceiling} & \sum_{h \in \mathcal{H}} w^{\mathrm{rt}}_{h} & \text{slack}\\
\hline
12 & \text{Storage buffer} & \sum_{h \in \mathcal{H}} o^{\mathrm{buf}}_{h} & \text{guardrail}\\
13 & \text{Backlog toward band centre} & \sum_{h \in \mathcal{H}} d^{\mathrm{ctr}}_{h} & \text{guardrail}\\
\hline
14 & \text{Fewest movements} & \sum_{q,i,j} X_{q,i \to j} & \text{preference}\\
15 & \text{Park spare people in Support} & -\sum_{q} P_{q,\mathrm{SP}} & \text{preference}
\end{array}
$$

Two readings are easy to get wrong. **Rank 15 carries a minus sign**, so it *maximises*
people parked in Support - a tie-break rather than a cost, since Support produces nothing,
and it is what makes the output deterministic run to run. And **rank 14 carries a scalar
movement cost** which cannot change which solution is chosen: a single term in a single rank
scales the objective without reordering it.

The ordering of ranks 5-10 is why a backlog running *below* its floor is the cheapest thing
on the ladder to spend.

In [15]:
show_source(model._build_ladder)

`_build_ladder` &nbsp;&mdash;&nbsp; `model.py:561`, 40 lines

def _build_ladder(m: Model, xsum) -> list[Tier]:
    """The objective tiers, highest priority first.

    Within a family the current shift outranks everything later. A guardrail is a loose
    version of a family, settling which breached family gets attention first.
    """
    p = m.params
    now = [h for h in p.hours if p.tier_of_hour[h] == 0]
    later = [h for h in p.hours if p.tier_of_hour[h] == 1]
    cpt_now = [c for c in p.cpts if p.cpt_tier[c] == 0]
    cpt_later = [c for c in p.cpts if p.cpt_tier[c] == 1]

    def over(d: dict, keys) -> Any:
        return xsum(d[k] for k in keys if k in d)

    moves = xsum(
        cfg.THRESHOLDS["move_cost"] * v for v in m.X.values()
    )
    support = xsum(-m.P[(q, cfg.SP)] for q in p.quarters)

    return [
        Tier("storage_now", over(m.o_storage, now), "Inbound storage (this shift)"),
        Tier("storage_later", over(m.o_storage, later), "Inbound storage (later)"),
        Tier("cpt_now", over(m.g_cpt, cpt_now), "Carrier cut-off risk (this shift)"),
        Tier("cpt_later", over(m.g_cpt, cpt_later), "Carrier cut-off risk (later)"),
        Tier("ob_bl_max_now", over(m.v_minus, now), "Outbound backlog too high (this shift)"),
        Tier("ob_bl_max_later", over(m.v_minus, later), "Outbound backlog too high (later)"),
        Tier("eod_max", m.e_eod_max, "End-of-day backlog above target"),
        Tier("ob_bl_min_now", over(m.v_plus, now), "Outbound backlog too low (this shift)"),
        Tier("ob_bl_min_later", over(m.v_plus, later), "Outbound backlog too low (later)"),
        Tier("eod_min", m.e_eod_min, "End-of-day backlog below target"),
        Tier("returns_bl", over(m.w_rt, p.hours), "Returns backlog ceiling"),
        # --- bottom guardrails ---------------------------------------------
        Tier("storage_buffer", over(m.o_storage_buffer, p.hours),
             "Inbound storage: buffer under the ceiling", kind="guardrail"),
        Tier("ob_bl_centre", over(m.d_centre, p.hours),
             "Outbound backlog: toward band centre", kind="guardrail"),
        Tier("moves", moves, "Minimise movements", kind="preference"),
        Tier("support", support, "Park spare people in Support", kind="preference"),
    ]

In [16]:
pd.DataFrame(
    [{"rank": i, "tier": t.name, "label": t.label, "kind": t.kind}
     for i, t in enumerate(m.tiers, 1)]
).set_index("rank")

,tier,label,kind
rank,,,
1,storage_now,Inbound storage (this shift),slack
2,storage_later,Inbound storage (later),slack
3,cpt_now,Carrier cut-off risk (this shift),slack
4,cpt_later,Carrier cut-off risk (later),slack
5,ob_bl_max_now,Outbound backlog too high (this shift),slack
6,ob_bl_max_later,Outbound backlog too high (later),slack
7,eod_max,End-of-day backlog above target,slack
8,ob_bl_min_now,Outbound backlog too low (this shift),slack
9,ob_bl_min_later,Outbound backlog too low (later),slack


---
## 3. Solve, freeze, continue

For each rank $k$ in order:

$$
v_k = \min f_k,
\qquad\text{then add}\qquad
f_k \le v_k + \varepsilon_k
$$

That added constraint is the whole mechanism. Once rank $k$'s attained value is frozen, every
rank below it optimises *subject to* it, so it cannot trade against it however large the
numbers get.

Below is the production implementation, then the same loop stepped by hand so each rank's
attained value and freeze are visible.

In [17]:
show_source(model.solve_lexicographic)

`solve_lexicographic` &nbsp;&mdash;&nbsp; `model.py:166`, 22 lines

def solve_lexicographic(backend: Backend, tiers: list[Tier]) -> LadderResult:
    """Solve the tiers in strict priority order, highest priority first."""
    result = LadderResult()
    _check_licence_size(backend)
    for tier in tiers:
        backend.set_objective(tier.expr)
        status = backend.solve()
        if status is not Status.OPTIMAL:
            raise TierInfeasible(tier.name, status)
        v = backend.value(tier.expr)
        result.attained[tier.name] = v
        result.order.append(tier.name)
        result.labels[tier.name] = tier.label
        result.kinds[tier.name] = tier.kind
        rel = (
            cfg.FREEZE_EPS_REL_GUARDRAIL
            if tier.kind == "guardrail"
            else cfg.FREEZE_EPS_REL
        )
        slack = cfg.FREEZE_EPS_ABS + rel * abs(v)
        backend.add(tier.expr <= v + slack, f"freeze_{tier.name}")
    return result

In [18]:
# The same loop as above, instrumented. This is the only place in either notebook that
# restates library code, and it is restated because the per-rank state is the point.
rows = []
for k, tier in enumerate(m.tiers, 1):
    backend.set_objective(tier.expr)
    status = backend.solve()
    assert status is model.Status.OPTIMAL, f"rank {k} ({tier.name}) did not solve: {status}"

    v = backend.value(tier.expr)

    # Guardrail ranks freeze LOOSELY - five orders of magnitude looser. A deadbanded
    # guardrail is degenerate, so its attained value only reproduces to about a percent.
    rel = (cfg.FREEZE_EPS_REL_GUARDRAIL if tier.kind == "guardrail"
           else cfg.FREEZE_EPS_REL)
    slack = cfg.FREEZE_EPS_ABS + rel * abs(v)
    backend.add(tier.expr <= v + slack, f"walkthrough_freeze_{tier.name}")

    rows.append({"rank": k, "tier": tier.name, "kind": tier.kind,
                 "attained": round(v, 3), "freeze_eps": f"{slack:.2e}",
                 "frozen_at": f"<= {v + slack:,.3f}"})

ladder = pd.DataFrame(rows).set_index("rank")
ladder

,tier,kind,attained,freeze_eps,frozen_at
rank,,,,,
1,storage_now,slack,0.000,1.00e-04,<= 0.000
2,storage_later,slack,0.000,1.00e-04,<= 0.000
3,cpt_now,slack,0.000,1.00e-04,<= 0.000
4,cpt_later,slack,0.000,1.00e-04,<= 0.000
5,ob_bl_max_now,slack,0.000,1.00e-04,<= 0.000
6,ob_bl_max_later,slack,0.000,1.00e-04,<= 0.000
7,eod_max,slack,0.000,1.00e-04,<= 0.000
8,ob_bl_min_now,slack,0.000,1.00e-04,<= 0.000
9,ob_bl_min_later,slack,0.000,1.00e-04,<= 0.000


---
## 3b. What the ladder did

The table above is the **after** solve, so a rank reading zero there has usually been *fixed*
rather than never breached. The pair is what carries the result, and getting it requires
solving the same model twice - once with movements pinned to zero.

This is also the cheapest demonstration that the priority order is real: watch which ranks
go to zero and which are *spent* to get them there.

In [19]:
# What the ladder actually DID. The table above is the AFTER solve, so a rank reading zero
# there has usually been fixed rather than never breached - which is the point of the whole
# exercise and is invisible without the pair.
#
# So: build the same model with movements pinned to zero, run the same ladder, and compare.
pin_backend = model.make_backend(label="no_move_baseline")
pinned = model.build(pin_backend, p, freeze_moves=True)
pin = model.solve_lexicographic(pin_backend, pinned.tiers)

pair = pd.DataFrame([
    {"rank": k, "tier": t.name, "kind": t.kind,
     "before": round(pin.attained[t.name], 1),
     "after": round(ladder.loc[k, "attained"], 1)}
    for k, t in enumerate(m.tiers, 1)
]).set_index("rank")

slacks = pair.query("kind == 'slack' and (before > 1e-6 or after > 1e-6)")
print("operational ranks with a non-zero value on either side:\n")
print(slacks.to_string() if len(slacks) else "  none on either side")

g = pair.query("kind == 'guardrail'")
print(f"\nrisk carried, both guardrails:  before {g['before'].sum():,.0f}"
      f"  ->  after {g['after'].sum():,.0f} units of distance")
print("  (a position, not a breach - non-zero here is not a defect)")
pair

operational ranks with a non-zero value on either side:

                 tier   kind   before  after
rank                                        
6     ob_bl_max_later  slack  22661.6    0.0

risk carried, both guardrails:  before 84,919  ->  after 1,806 units of distance
  (a position, not a breach - non-zero here is not a defect)


,tier,kind,before,after
rank,,,,
1,storage_now,slack,0.0,0.0
2,storage_later,slack,0.0,0.0
3,cpt_now,slack,0.0,0.0
4,cpt_later,slack,0.0,0.0
5,ob_bl_max_now,slack,0.0,0.0
6,ob_bl_max_later,slack,22661.6,0.0
7,eod_max,slack,0.0,0.0
8,ob_bl_min_now,slack,0.0,0.0
9,ob_bl_min_later,slack,0.0,0.0


---
## 4. Why the tolerances are what they are

Four constants in `config.py` that look like noise and are not. Each was arrived at by a
failure, and each is invisible to the engineered scenarios - every one was found by running
generated days.

$$
\varepsilon_k = 10^{-4} + \rho^{\mathrm{frz}}_{k}\,|v_k|,
\qquad
\rho^{\mathrm{frz}}_{k} =
\begin{cases}
10^{-7} & \text{slack and preference ranks}\\
10^{-2} & \text{guardrail ranks}
\end{cases}
$$

**The solver gap has to be tighter than the freeze.** The ladder freezes each attained value
as a constraint for the next solve, so that value has to be reproducible *across solves*. At
the 1e-4 relative gap solvers default to, a rank attaining 169,061 can come back ~17 units
different and the frozen constraint then reads as infeasible. The symptom appears several
ranks later than the cause - typically `TierInfeasible` on `moves`, a rank that cannot fail
on its own.

**The guardrail tolerance is scoped, not global.** Five orders of magnitude looser, applied
to guardrail ranks only. A deadbanded guardrail is *degenerate*: inside the deadband the term
sits at its lower bound, so hours already fine constrain nothing and many solutions attain
the identical value. With one tight epsilon everywhere, one generated seed fails outright.
But promoting 1e-2 to a global value would let a lower rank buy a 1% degradation of a carrier
cut-off breach - immaterial for a distance-from-target, thousands of units of real risk for a
breach.

In [20]:
for name in ("SOLVER_GAP_REL", "SOLVER_GAP_ABS", "FREEZE_EPS_ABS",
             "FREEZE_EPS_REL", "FREEZE_EPS_REL_GUARDRAIL"):
    print(f"  cfg.{name:<26} = {getattr(cfg, name):g}")

print(f"\nfreeze epsilon on a rank attaining 169,061:")
for kind, rel in (("slack", cfg.FREEZE_EPS_REL),
                  ("guardrail", cfg.FREEZE_EPS_REL_GUARDRAIL)):
    print(f"  {kind:<10} {cfg.FREEZE_EPS_ABS + rel * 169_061:,.4f} units of slack")
print(f"\nsolver's own gap at that magnitude: "
      f"{cfg.SOLVER_GAP_REL * 169_061:.2e} units - comfortably below both")

  cfg.SOLVER_GAP_REL             = 1e-09
  cfg.SOLVER_GAP_ABS             = 1e-06
  cfg.FREEZE_EPS_ABS             = 0.0001
  cfg.FREEZE_EPS_REL             = 1e-07
  cfg.FREEZE_EPS_REL_GUARDRAIL   = 0.01

freeze epsilon on a rank attaining 169,061:
  slack      0.0170 units of slack
  guardrail  1,690.6101 units of slack

solver's own gap at that magnitude: 1.69e-04 units - comfortably below both


---
## 5. The two non-linearities

$V = \max(V^{\mathrm{raw}},0)$ and $Y = \min(\mathrm{Hcap},V)$ are piecewise linear, which is
what makes this a mixed-integer program. Both are handed to the solver as **general
constraints**, which it supports natively - nothing is reformulated by hand.

A solver without them needs a big-M disjunction and one binary per $\max$, plus one per
$\min$ argument, at every hour of every department. A second backend doing exactly that was
carried through development and attained identical objective values on every scenario, which
is evidence about the *model* rather than about one encoding of it. It was retired once that
was recorded, and ~19% of the variables and ~75% of the constraints went with it.

Note what `BIG_M` is still for. It is **not** a linearisation constant any more: it is the
lower bound on $V^{\mathrm{raw}}$ - the one variable allowed below zero, which therefore needs
a finite bound that cannot bind - and a sanity check on the inputs.

In [21]:
show_source(model.GurobiBackend.add_min)

`GurobiBackend.add_min` &nbsp;&mdash;&nbsp; `model.py:96`, 3 lines

def add_min(self, target, args, name):
        """target == min(args), exactly."""
        self._m.addGenConstrMin(target, args, name=name)

In [22]:
show_source(model.GurobiBackend.add_max_zero)

`GurobiBackend.add_max_zero` &nbsp;&mdash;&nbsp; `model.py:100`, 3 lines

def add_max_zero(self, target, arg, name):
        """target == max(arg, 0), exactly, where `arg` may be negative."""
        self._m.addGenConstrMax(target, [arg], constant=0.0, name=name)

In [23]:
print(f"cfg.BIG_M = {cfg.BIG_M:,.0f}")
print(f"  used as the lower bound on V_raw, and as a data sanity check in data.build")
print(f"  licence guard: {cfg.LICENCE_MAX_VARS} vars / {cfg.LICENCE_MAX_CONSTRS} cons")
nv, nc = backend.report_size()
print(f"  this model:   {nv} vars / {nc} cons"
      f"   ({nv / cfg.LICENCE_MAX_VARS:.0%} / {nc / cfg.LICENCE_MAX_CONSTRS:.0%} of the cap)")

cfg.BIG_M = 5,000,000
  used as the lower bound on V_raw, and as a data sanity check in data.build
  licence guard: 2000 vars / 2000 cons
  this model:   922 vars / 769 cons   (46% / 38% of the cap)


---
## 6. The replay identity

The claim every before/after chart depends on: solve the same model with every movement
pinned to zero, and it reproduces the **planned** backlog trajectory exactly. Not
approximately - to zero units.

Without it, a difference on any chart could be reconstruction drift rather than the
recommendation. This is the same property the deployed retrospective rests on, and it is the
first thing to check after touching anything in the flow block.

In [24]:
# A fresh backend, movements frozen - the BEFORE solve.
before_backend = model.make_backend(label="replay_check")
before = model.build(before_backend, p, freeze_moves=True)
model.solve_lexicographic(before_backend, before.tiers)

worst = 0.0
for d in (cfg.OB, cfg.IB, cfg.RT):
    for h in p.hours:
        got = before_backend.value(before.U[(d, h)])
        want = p.planned_backlog.get((d, h))
        if want is not None:
            worst = max(worst, abs(got - want))

print(f"departments checked: OB, IB, RT over {len(p.hours)} hours")
print(f"max |solved - planned| backlog: {worst:.9f} units")
assert worst < 1e-6, f"replay identity broken: {worst}"
print("\nOK  the no-move solve reproduces the plan")

departments checked: OB, IB, RT over 24 hours
max |solved - planned| backlog: 0.000000000 units

OK  the no-move solve reproduces the plan
